## Query execution and Error management

### Snowflake Query execution

Fist of all we need to run the generated SQL query on Snowflake. First, we will reconnect to Snowflake

In [31]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
import numpy as np
import pandas as pd

import json
from pathlib import Path

import anthropic

# Change folder to current directory of the Jupyter Notebook
current_path=os.getcwd()
os.chdir(current_path)

# Load environment variables from .env file
load_dotenv(override=True) # override=True para que no haya conflictos con la Claude API Key en el .env 

# Connecting to Claude API
client = anthropic.Anthropic()

# Connection parameters for establishing a Snowpark session (Snowflake)
# These environment variables must be set with real credentials before running
connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
    "schema": os.getenv("SNOWFLAKE_SCHEMA"),
}

# Create a Snowpark session using the connection parameters
session = Session.builder.configs(connection_params).create()

For testing purposes, I will run a sample query from the ones obtained fromm the tests of the previous notebook:

In [ ]:
# Running the SQL query in Snowflake
sample_query = 'SELECT\n    c.country,\n    COUNT(DISTINCT c.customer_id) AS number_members,\n    AVG(c.total_sales) AS avg_total_sales,\n    AVG(ARRAY_SIZE(c.visited_location_ids_array)) AS avg_visited_locations\nFROM tasty_bytes.analytics.customer_loyalty_metrics_v AS c\nGROUP BY 1\nORDER BY 3 DESC;'
df = session.sql(sample_query).collect()
df

[Row(COUNTRY='Germany', NUMBER_MEMBERS=15790, AVG_TOTAL_SALES=Decimal('3070.1820455985'), AVG_VISITED_LOCATIONS=Decimal('65.870044')),
 Row(COUNTRY='Spain', NUMBER_MEMBERS=8516, AVG_TOTAL_SALES=Decimal('2993.5349635979'), AVG_VISITED_LOCATIONS=Decimal('68.671090')),
 Row(COUNTRY='Brazil', NUMBER_MEMBERS=10757, AVG_TOTAL_SALES=Decimal('2879.1758854699'), AVG_VISITED_LOCATIONS=Decimal('64.105699')),
 Row(COUNTRY='South Africa', NUMBER_MEMBERS=4763, AVG_TOTAL_SALES=Decimal('2782.0845055637'), AVG_VISITED_LOCATIONS=Decimal('69.013017')),
 Row(COUNTRY='Sweden', NUMBER_MEMBERS=6192, AVG_TOTAL_SALES=Decimal('2703.9337047804'), AVG_VISITED_LOCATIONS=Decimal('59.636143')),
 Row(COUNTRY='Egypt', NUMBER_MEMBERS=5772, AVG_TOTAL_SALES=Decimal('2696.5808645184'), AVG_VISITED_LOCATIONS=Decimal('62.757450')),
 Row(COUNTRY='India', NUMBER_MEMBERS=15748, AVG_TOTAL_SALES=Decimal('2605.2570326391'), AVG_VISITED_LOCATIONS=Decimal('55.174625')),
 Row(COUNTRY='United Kingdom', NUMBER_MEMBERS=18644, AVG_TOTAL

This is the type of output we obtained originally after executing the query on Snowflake. But, we don't want to use this, since we need to present to the app and Claude a clean dataframe, which is why I will use the method `to_pandas()` instead of `collect()`: 

In [36]:
# Running the SQL query in Snowflake
sample_query = 'SELECT\n    c.country,\n    COUNT(DISTINCT c.customer_id) AS number_members,\n    AVG(c.total_sales) AS avg_total_sales,\n    AVG(ARRAY_SIZE(c.visited_location_ids_array)) AS avg_visited_locations\nFROM tasty_bytes.analytics.customer_loyalty_metrics_v AS c\nGROUP BY 1\nORDER BY 3 DESC;'
df = session.sql(sample_query).to_pandas()
df

,COUNTRY,NUMBER_MEMBERS,AVG_TOTAL_SALES,AVG_VISITED_LOCATIONS
0,Germany,15790,3070.182046,65.870044
1,Spain,8516,2993.534964,68.671090
2,Brazil,10757,2879.175885,64.105699
3,South Africa,4763,2782.084506,69.013017
4,Sweden,6192,2703.933705,59.636143
5,Egypt,5772,2696.580865,62.757450
6,India,15748,2605.257033,55.174625
7,United Kingdom,18644,2487.095621,55.747050
8,South Korea,7985,2478.185755,57.370319
9,Canada,28302,2455.356680,56.832945


Now, I have the dataframe prepared, but now it is necessary to convert it into something Claude can read for the next Claude API call, that will create a graph out of this results. In order to not waste too much tokens, the following transformations will be applied to the dataframe:
- **Save it into a CSV format**: The table will be converted into a CSV text format, since is the cheapeast way thaat Claude can read the data.
- **Rounding decimal columns**: For getting business insights, 6 decimals on sales metrics are unnecessaary. We can work with just 2 decimals in order to save tokens.

In [37]:
df = df.round(2) # Rounding the columns of the dataframe that are Decimal type
df

,COUNTRY,NUMBER_MEMBERS,AVG_TOTAL_SALES,AVG_VISITED_LOCATIONS
0,Germany,15790,3070.18,65.87
1,Spain,8516,2993.53,68.67
2,Brazil,10757,2879.18,64.11
3,South Africa,4763,2782.08,69.01
4,Sweden,6192,2703.93,59.64
5,Egypt,5772,2696.58,62.76
6,India,15748,2605.26,55.17
7,United Kingdom,18644,2487.10,55.75
8,South Korea,7985,2478.19,57.37
9,Canada,28302,2455.36,56.83


In [8]:
csv_text = df.to_csv(index=False) # Converting into CSV text
csv_text

'COUNTRY,NUMBER_MEMBERS,AVG_TOTAL_SALES,AVG_VISITED_LOCATIONS\r\nGermany,15790,3070.18,65.87\r\nSpain,8516,2993.53,68.67\r\nBrazil,10757,2879.18,64.11\r\nSouth Africa,4763,2782.08,69.01\r\nSweden,6192,2703.93,59.64\r\nEgypt,5772,2696.58,62.76\r\nIndia,15748,2605.26,55.17\r\nUnited Kingdom,18644,2487.1,55.75\r\nSouth Korea,7985,2478.19,57.37\r\nCanada,28302,2455.36,56.83\r\nPoland,14056,2433.33,55.81\r\nAustralia,15217,2199.91,49.33\r\nJapan,10117,2164.02,52.34\r\nFrance,14381,2113.85,48.62\r\nUnited States,46300,2093.8,50.89\r\n'

### Error Management

Even though it can be rare when Claude can give an output with the invalid syntax (no test case had a syntax error), it can still occur if the model hallucinates enough. To be prepared for this, an extra step on the pipeline will be created using another Claude API call that will review the code and fix it, up to 2 attempts.

Defining again the relevant functioins previously made for using the Claude API

In [32]:
def add_user_msg(msgs, txt):
    ##### Adds an user message to the history #####
    user_msg = {'role': 'user', 'content':txt}
    msgs.append(user_msg)

def add_assistant_msg(msgs, txt):
    ##### Adds an assistant (AI) message to the history #####
    assistant_msg = {'role': 'assistant', 'content':txt}
    msgs.append(assistant_msg)

def chat_and_tokens(msgs, model, max_tokens=1000, use_cache=True, system_prompt=None, output_format=None):
     ##### Creates a chat with the corresponding message history, and outputs the input and output tokens used #####
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': msgs
    }
    if system_prompt:
        if use_cache: # This will execute if cache usage is needed
            params['system'] = [
                {
                    "type": "text",
                    "text": system_prompt,
                    "cache_control": {"type": "ephemeral"}  # Configuring the system prompt to be written in the cache. Lasts 5 minutes by default
                }
            ]
        else: # This will execute if cache usage is not needed
            params['system'] = system_prompt
    if output_format:
        params['output_config'] = {'format': output_format}

    output = client.messages.create(**params)
    # Using an array for saving the tokens used, so for future usages, we can add up the tokens used
    tokens_used = np.array([output.usage.input_tokens, output.usage.output_tokens, output.usage.cache_creation_input_tokens, output.usage.cache_read_input_tokens])
    # Joining all text blocks, and checking if the block is a TextBlock
    return "".join(b.text for b in output.content if b.type == "text"), tokens_used

Defining the function that will ask Claude to fix the SQL query:

In [33]:
def fix_bug(ai_query_bugged, error_msg, system_prompt, messages):
    """
    Fixes the SQL code using a Claude API call.

    Parameters
    ----------
    ai_query_bugged : str
        The SQL query written by Claude that contains an error
    error_msg : str
        The error message raised by Snowpark
    system_prompt : str
        System prompt for Claude, with all the specifications and database documentation.
    messages : list[dict]
        Message history of user-assistant conversation.

    Returns
    ----------
    ai_answer : str
        Claude's answer wwith the code fixed as a string, but with a JSON structure.
    messages : list[dict]
        Message history of user-assistant conversation.
    """ 

    prompt = f""" 
The SQL query below was generated earlier in this conversation and failed when executed in Snowflake.

<failed_query>
{ai_query_bugged}
</failed_query>

<snowflake_error>
{error_msg}
</snowflake_error>

Fix the cause of the error, but also review the query and fix other problems you might find. Keep the original intent, the schema rules and the SQL style. 
Reply with the usual JSON object, always with "question_type": "sql" (never "clarify"); in "explanation", state in one sentence what was wrong and what you changed.
    """

    # Creating a JSON Schema that Claude will use for the output
    sql_schema = { # Now, the only possible output schema could be the one where it outputs the SQL query
        "type": "json_schema",
        "schema": { 
            "type": "object",
            "properties": {
                "question_type": {"type": "string", "const": "sql"},
                "query": {
                    "type": "string",
                    "description": "Executable SQL Query that answers the question.",
                },
                "explanation": {
                    "type": "string",
                    "description": "Brief explanation of the query of no more than 3 sentences.",
                },
            },
            "required": ["question_type", "query", "explanation"],
            "additionalProperties": False,
        },
    }

    add_user_msg(messages, prompt) # Adding the new prompt to the message history

    ai_answer, tokens = chat_and_tokens( # Calling Claude API to write the SQL query
        messages,
        model='claude-sonnet-5-5',
        max_tokens=2000,
        system_prompt=system_prompt,
        output_format=sql_schema,
    )

    add_assistant_msg(messages, ai_answer) # Adding the user question prompt to the message history
    ai_answer = json.loads(ai_answer) # Converting the string generated by Claude into an JSON (dict)

    return ai_answer, tokens, messages

In [35]:
# TODO: hacer una función con un prompt breve pero conciso para que corrija el código para que el output sea el código SQL corregido. 
# A Claude le darpe como input el SQL y la breve explicación de lo que quiso hacer Claude originalmente. NO SÉ si ponerle la documentación o no

# Using the same sample query as before, but introducing artificial errors on it (changed "DISTINCT" to "DISTINT" and "customer_loyalty_metrics_v" to "customer_loyalty_metrics")
sample_query = 'SELECT\n    c.country,\n    COUNT(DISTINT c.customer_id) AS number_members,\n    AVG(c.total_sales) AS avg_total_sales,\n    AVG(ARRAY_SIZE(c.visited_location_ids_array)) AS avg_visited_locations\nFROM tasty_bytes.analytics.customer_loyalty_metrics AS c\nGROUP BY 1\nORDER BY 3 DESC;'

# Loading system prompt. Here, the system prompt should be the same as the one used before, so that Claude can read it from the cache for less token cost.
sys_prompt = Path('claude_prompt_fully_improved.md').read_text(encoding='utf-8') 

msg_hist = [] # FOr this test, I will not include the full message history, but on the final app, it should read all of the previous conversation with Claude

print(sample_query, '\n')

for attempt in [1,2,3]: # Maximum 2 more attempts (first execution + 2 extra attempts)
    try:
        query_result = session.sql(sample_query).to_pandas()
    except Exception as e:
        if attempt == 3: # If maximum attempts reached, raise an error and stop the streamlit program
            print('Maximimum number of attempts reached, Claude could not fix the SQL query') # TODO
            break
        # If an error occurs and there are still enough attempts, the fix_bug() function will execute
        error_message = str(e) # REtrieving the error explicitly for Claude
        ai_answer, used_tokens_now, msg_hist = fix_bug(ai_query_bugged=sample_query, error_msg=error_message, system_prompt=sys_prompt, messages=msg_hist)
        # used_tokens += used_tokens_now
    else: 
        # If no errors where found, there is no need to fix the query and we can break the for loop
        break

print(ai_answer['query'], ai_answer['explanation'], sep='\n\n')

SELECT
    c.country,
    COUNT(DISTINT c.customer_id) AS number_members,
    AVG(c.total_sales) AS avg_total_sales,
    AVG(ARRAY_SIZE(c.visited_location_ids_array)) AS avg_visited_locations
FROM tasty_bytes.analytics.customer_loyalty_metrics AS c
GROUP BY 1
ORDER BY 3 DESC; 

Maximimum number of attempts reached, Claude could not fix the SQL query
SELECT
    c.country,
    COUNT(DISTINCT c.customer_id) AS number_members,
    AVG(c.total_sales) AS avg_total_sales,
    AVG(ARRAY_SIZE(c.visited_location_ids_array)) AS avg_visited_locations
FROM tasty_bytes.analytics.customer_loyalty_metrics_v AS c
GROUP BY 1
ORDER BY 3 DESC;

The query had a typo (DISTINT instead of DISTINCT) and referenced a non-existent view (customer_loyalty_metrics instead of customer_loyalty_metrics_v), so I fixed both; it now returns, per customer residence country, the number of loyalty members, average total sales and average distinct locations visited, ordered by average total sales.


## Graphs generation with Claude

In [ ]:
# TODO: Ver si usasré las gráficas integradas de streamlit o gráficas "altair"